# IPL Intelligence — 01. Data Quality & Tournament Overview

**Purpose:** establish data trust before interpreting cricket performance. This notebook audits coverage, missingness, duplicates, season trends and basic scoring patterns.

**Source caveat:** this Cricsheet ball-by-ball archive records delivery outcomes and dismissals, but not shot labels, pitch maps, or wagon-wheel coordinates. Do not infer pull shots or exact shot direction from these files.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "deliveries.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
df = pd.read_csv(DATA / "deliveries.csv", low_memory=False)
matches = pd.read_csv(DATA / "matches.csv", low_memory=False)
for c in ["start_date"]:
    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
for c in ["runs_off_bat","extras","total_runs","legal_ball","is_wicket","is_bowler_credited_wicket","over_number"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["season"] = df["season"].astype(str)
print(f"Deliveries: {len(df):,} | Matches: {df.match_id.nunique():,} | Seasons: {df.season.nunique()}")

## 1. Schema and sample

In [ ]:
display(df.head())
print("\nColumns:", list(df.columns))
display(df.dtypes.astype(str).to_frame("dtype"))

## 2. Core data-quality checks

In [ ]:
quality = pd.DataFrame({
    "check": ["rows", "unique matches", "missing match_id", "missing striker", "missing bowler", "exact duplicate rows", "missing start_date", "missing season"],
    "value": [len(df), df.match_id.nunique(), df.match_id.isna().sum(), df.striker.isna().sum(), df.bowler.isna().sum(), df.duplicated().sum(), df.start_date.isna().sum(), df.season.isna().sum()]
})
display(quality)
print("Date range:", df.start_date.min(), "to", df.start_date.max())
print("Season labels:", sorted(df.season.dropna().unique(), key=str))

## 3. Deliveries and scoring by season

In [ ]:
season = df.groupby("season", as_index=False).agg(
    deliveries=("match_id", "size"), matches=("match_id", "nunique"),
    total_runs=("total_runs", "sum"), batter_runs=("runs_off_bat", "sum"),
    wickets=("is_wicket", "sum"), legal_balls=("legal_ball", "sum")
)
season["run_rate"] = np.where(season.legal_balls > 0, 6*season.total_runs/season.legal_balls, np.nan)
display(season)
fig, ax = plt.subplots(figsize=(12,5)); ax.plot(season.season, season.total_runs, marker="o"); ax.set_title("Recorded IPL runs by season"); ax.set_xlabel("Season"); ax.set_ylabel("Runs"); ax.tick_params(axis="x", rotation=60); fig.tight_layout()

## 4. Runs and wickets by innings phase

In [ ]:
phase_order = ["Powerplay (1–6)", "Middle (7–15)", "Death (16+)"]
phase = df.groupby("phase", as_index=False).agg(runs=("total_runs","sum"), wickets=("is_wicket","sum"), legal_balls=("legal_ball","sum"))
phase["run_rate"] = np.where(phase.legal_balls>0, 6*phase.runs/phase.legal_balls, np.nan)
phase["phase"] = pd.Categorical(phase.phase, categories=phase_order, ordered=True)
phase = phase.sort_values("phase"); display(phase)
fig, ax = plt.subplots(figsize=(8,4)); ax.bar(phase.phase.astype(str), phase.run_rate); ax.set_ylabel("Runs per over"); ax.set_title("Run rate by phase"); ax.tick_params(axis="x", rotation=15); fig.tight_layout()

## Interpretation checklist
- Are any seasons much smaller because coverage is incomplete?
- Do match counts and delivery counts change abruptly?
- Does the run-rate formula use legal balls rather than raw deliveries?
- Treat this archive as the source of available recorded matches, not necessarily a complete official historical census.